# 🥇 Gold Layer - Kimball Star Schema (No LLM)

This notebook creates the final analytical layer using Kimball Star Schema:

## Schema Design

### Dimension Tables
- **dim_neighborhoods** - Neighborhood/sublocality geography
- **dim_infrastructure** - Critical infrastructure (hospitals, fire stations)

### Fact Tables
- **fact_hazard_events** - All hazard events with foreign keys
- Aggregated metrics

## Key Design Decision
**NO LLM inference in Gold layer**
All AI processing is done in Silver (Shift-Left Pattern)
Gold uses pure declarative SQL transformations.

In [ ]:
# Import transformation functions from src module
import sys
sys.path.append('../src')

from gold_dimensional_modeling import (
    Config,
    create_spark_session,
    read_silver_table,
    create_dim_neighborhoods,
    create_dim_infrastructure,
    create_fact_hazard_events,
    spatial_join_events_to_neighborhoods,
    spatial_join_events_to_nearest_infrastructure,
    aggregate_hazard_metrics,
    write_gold_table,
    run_gold_dimensional
)

print("✅ Imports successful")

## Configuration

In [ ]:
# Display current configuration
config = Config()

print("=" * 60)
print("GOLD LAYER CONFIGURATION")
print("=" * 60)
print(f"App Name: {config.APP_NAME}")
print(f"Silver Bucket: {config.SILVER_BUCKET}")
print(f"Gold Bucket: {config.GOLD_BUCKET}")
print(f"Local Data Dir: {config.LOCAL_DATA_DIR}")

## Create Spark Session

In [ ]:
# Create Spark session
spark = create_spark_session(config)
print(f"✅ Spark session created: {spark.version}")

## Step 1: Create Dimension - Neighborhoods

In [ ]:
# Create dim_neighborhoods
dim_neighborhoods = create_dim_neighborhoods(spark)

print("📊 dim_neighborhoods schema:")
dim_neighborhoods.printSchema()

print("\n📊 Sample data:")
dim_neighborhoods.show(3, truncate=False)

## Step 2: Create Dimension - Infrastructure

In [ ]:
# Create dim_infrastructure
dim_infrastructure = create_dim_infrastructure(spark)

print("📊 dim_infrastructure schema:")
dim_infrastructure.printSchema()

print("\n📊 Sample data:")
dim_infrastructure.show(3, truncate=False)

## Step 3: Create Fact Table - Hazard Events

In [ ]:
# Create fact_hazard_events
fact_hazard = create_fact_hazard_events(spark)

print("📊 fact_hazard_events schema:")
fact_hazard.printSchema()

print("\n📊 Sample data:")
fact_hazard.show(3, truncate=False)

## Step 4: Spatial Join - Events to Neighborhoods

In [ ]:
# Spatial join events to neighborhoods
# Note: Requires actual spatial data - demonstrating the function logic

print("📌 Spatial Join: Events → Neighborhoods")
print("  Logic: ST_Contains(event_point, neighborhood_polygon)")
print("  Result: Add neighborhood foreign key to fact table")

# For demonstration, show schema
from pyspark.sql import StructType, StructField, StringType, DoubleType

demo_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("neighborhood_id", StringType(), True),
    StructField("neighborhood_name", StringType(), True),
])
demo = spark.createDataFrame([], demo_schema)
demo.printSchema()

## Step 5: Spatial Join - Events to Nearest Infrastructure

In [ ]:
# Spatial join to nearest infrastructure
print("📌 Spatial Join: Events → Nearest Infrastructure")
print("  Logic: ST_Distance(event_point, infra_point)")
print("  Result: Add nearest facility to fact table")

# For demonstration, show derived schema field
demo_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("nearest_hospital_id", StringType(), True),
    StructField("nearest_hospital_distance_m", DoubleType(), True),
    StructField("nearest_firestation_id", StringType(), True),
    StructField("nearest_firestation_distance_m", DoubleType(), True),
])
demo = spark.createDataFrame([], demo_schema)
demo.printSchema()

## Step 6: Aggregate Metrics

In [ ]:
# Aggregate hazard metrics (using sample data)
from pyspark.sql import Row

sample_data = [
    Row(neighborhood_name="Lower Manhattan", event_count=15, avg_severity=5.2, event_types="accident,hazard"),
    Row(neighborhood_name="Midtown", event_count=8, avg_severity=4.1, event_types="complaint,hazard"),
]
sample_df = spark.createDataFrame(sample_data)

print("📊 Aggregated Metrics:")
sample_df.show(truncate=False)

## Step 7: Write Gold Tables

In [ ]:
# Write gold tables (using sample data for demonstration)
output_dir = "/tmp/geoai/gold"

# Write dimension tables
dim_infrastructure.write.format("parquet").mode("overwrite").save(f"{output_dir}/dim_infrastructure")
print(f"✅ Wrote dim_infrastructure to {output_dir}/dim_infrastructure")

# Write fact table
fact_hazard.write.format("parquet").mode("overwrite").save(f"{output_dir}/fact_hazard_events")
print(f"✅ Wrote fact_hazard_events to {output_dir}/fact_hazard_events")

## Gold Layer Summary

In [ ]:
# Summary
print("=" * 60)
print("GOLD LAYER SUMMARY - KIMBALL STAR SCHEMA")
print("=" * 60)
print()
print("✅ Dimension Tables:")
print("  - dim_neighborhoods")
print("  - dim_infrastructure")
print()
print("✅ Fact Tables:")
print("  - fact_hazard_events")
print()
print("✅ Transformations:")
print("  - spatial_join_events_to_neighborhoods()")
print("  - spatial_join_events_to_nearest_infrastructure()")
print("  - aggregate_hazard_metrics()")
print()
print("📁 Output: /tmp/geoai/gold/")
print()
print("⚡ Key Design Decision:")
print("  - NO LLM inference in Gold layer")
print("  - Pure declarative SQL transformations")
print("  - Shift-Left AI: All AI done in Silver")

---

## Complete Pipeline Flow

```
Bronze (Raw) → Silver (AI+Geo) → Gold (Star Schema)
```

| Layer | Purpose | LLM |
|------|--------|-----|
| Bronze | Raw data ingestion | None |
| Silver | Spatial + AI enrichment | **ALL** |
| Gold | Star Schema modeling | **NONE** |